# Static vs. Dynamic Analysis (False Negatives Identification)

In [1]:
import dpkt
import socket
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(f"{BASE_DIR} does not exist")
DATA_DIR = os.path.join(BASE_DIR, 'data')
DATASET_DIR = os.path.join(DATA_DIR, '02_octopus')
EMULATOR_OUT_DIR = os.path.join(DATASET_DIR, 'emulator_out')

In [2]:
# Loading the static analysis parquet file
STATIC_ANALYSIS_DATAFRAME_PATH = os.path.join(DATASET_DIR,"static_analysis/static_analysis.parquet")
if os.path.exists(STATIC_ANALYSIS_DATAFRAME_PATH):
    print("Loading static analysis data...")
    df_static = pd.read_parquet(STATIC_ANALYSIS_DATAFRAME_PATH)

else:
    raise FileNotFoundError(f"{STATIC_ANALYSIS_DATAFRAME_PATH} does not exist")

Loading static analysis data...


In [3]:
print(df_static.columns)

Index(['_id', '_cls', 'report_date', 'android_app_id_reference',
       'scanner_version', 'scanner_name', 'scan_status', 'error', 'trackers',
       'application.handle', 'application.version_name',
       'application.version_code', 'application.uaid', 'application.name',
       'application.permissions', 'application.libraries', 'apk.path',
       'apk.checksum', 'os_vendor', 'version_detected', 'firmware_id',
       'filename', 'partition_name', 'tracker_names'],
      dtype='object')


In [4]:
df_static.head()

,_id,_cls,report_date,android_app_id_reference,scanner_version,scanner_name,scan_status,error,trackers,application.handle,...,application.permissions,application.libraries,apk.path,apk.checksum,os_vendor,version_detected,firmware_id,filename,partition_name,tracker_names
index,,,,,,,,,,,,,,,,,,,,,
960,6a4f4eb122324e493b7a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:02.498000,68b105d665e2ad36cf065848,1.3.12,Exodus,completed,None,[],com.android.bookmarkprovider,...,[],[],/var/www/file_store/01_file_storage/dc63ac8b-3...,b5d8898df49761cf62820f07d8f784b7ad26a147ad1e8a...,Google,12,68b1075d65e2ad36cf0776d5,BookmarkProvider.apk,system,[]
961,6a4f4eb1fa7e8ee5127a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:01.800000,68b105d665e2ad36cf065844,1.3.12,Exodus,completed,None,[],com.android.bluetoothmidiservice,...,"[""android.permission.BLUETOOTH_SCAN"",""android....",[],/var/www/file_store/01_file_storage/dc63ac8b-3...,911cdf6c13fd0b3d3c432c661ee1dab4edbd8cc1bed0cc...,Google,12,68b1075d65e2ad36cf0776d5,BluetoothMidiService.apk,system,[]
962,6a4f4eb256df774d197a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:03.001000,68b105d665e2ad36cf06582e,1.3.12,Exodus,completed,None,[],com.android.dreams.basic,...,[],[],/var/www/file_store/01_file_storage/dc63ac8b-3...,5a0900ec7b89f0ac562506e6ddc6b62c3ec52a532bec40...,Google,12,68b1075d65e2ad36cf0776d5,BasicDreams.apk,system,[]
963,6a4f4eb28147f5ecd97a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:02.704000,68b105d765e2ad36cf065944,1.3.12,Exodus,completed,None,[],com.android.providers.partnerbookmarks,...,[],[],/var/www/file_store/01_file_storage/dc63ac8b-3...,67d14c800a89b54e33ffa1979a23f7c47dd1a6db18b42f...,Google,12,68b1075d65e2ad36cf0776d5,PartnerBookmarksProvider.apk,system,[]
964,6a4f4eb31aea578a3f7a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:05.799000,68b105d765e2ad36cf065892,1.3.12,Exodus,completed,None,[],com.android.htmlviewer,...,[],[],/var/www/file_store/01_file_storage/dc63ac8b-3...,3c2dddb9262839eab44ac89b6f20678734a7126aa6fb1b...,Google,12,68b1075d65e2ad36cf0776d5,HTMLViewer.apk,system,[]


In [5]:
# Loading the dynamic analysis parquet file
DYNAMIC_ANALYSIS_DATAFRAME_PATH = os.path.join(DATASET_DIR,"delta/app_traffic.parquet")
if os.path.exists(DYNAMIC_ANALYSIS_DATAFRAME_PATH):
    print("Loading dynamic analysis data...")
    df_dynamic = pd.read_parquet(DYNAMIC_ANALYSIS_DATAFRAME_PATH)
    df_dynamic['firmware_id'] = df_dynamic['firmware_id'].astype(str).str.split('_').str[0].str.strip()
else:
    raise FileNotFoundError(f"{DYNAMIC_ANALYSIS_DATAFRAME_PATH} does not exist")

Loading dynamic analysis data...


In [6]:
print(df_dynamic.columns)

Index(['timestamp', 'firmware_id', 'uid', 'src_ip', 'dst_ip', 'src_port',
       'dst_port', 'protocol', 'length', 'real_domain', 'package_name'],
      dtype='object')


In [7]:
df_dynamic.head()

,timestamp,firmware_id,uid,src_ip,dst_ip,src_port,dst_port,protocol,length,real_domain,package_name
0,1.786567e+09,68afd6e865e2ad36cf1c1b8a,1020,10.0.2.15,224.0.0.251,5353.0,5353.0,UDP,527,224.0.0.251,UID 0 (OS) - Unknown Package
1,1.786567e+09,68afd6e865e2ad36cf1c1b8a,1020,10.0.2.15,224.0.0.251,5353.0,5353.0,UDP,527,224.0.0.251,UID 0 (OS) - Unknown Package
2,1.786567e+09,68afd6e865e2ad36cf1c1b8a,1020,10.0.2.15,224.0.0.251,5353.0,5353.0,UDP,527,224.0.0.251,UID 0 (OS) - Unknown Package
3,1.786567e+09,68afd6e865e2ad36cf1c1b8a,1020,10.0.2.15,224.0.0.251,5353.0,5353.0,UDP,527,224.0.0.251,UID 0 (OS) - Unknown Package
4,1.786567e+09,68afd6e865e2ad36cf1c1b8a,1020,10.0.2.15,224.0.0.251,5353.0,5353.0,UDP,527,224.0.0.251,UID 0 (OS) - Unknown Package


In [8]:
unique_domains = df_dynamic['real_domain'].dropna().unique()

print(unique_domains)

['224.0.0.251' '224.0.0.22' '10.0.2.3' '216.239.35.12' 'www.gstatic.com'
 'www.google.com' '127.0.0.1' 'youtubei.googleapis.com' 'www.youtube.com'
 'm.google.com' 'www.googleadservices.com' 'www.googletagmanager.com'
 '192.178.156.188' 'alt4-mtalk.google.com' 'ad.doubleclick.net'
 'android.googleapis.com' 'telephonyspamprotect-pa.googleapis.com'
 'instantmessaging-pa.googleapis.com' 'alt2-mtalk.google.com'
 'app-measurement.com' 'clientservices.googleapis.com'
 'android.clients.google.com' 'dl.google.com' 'ssl.gstatic.com'
 'pagead2.googlesyndication.com' 'clients4.google.com'
 'verizon-services.com' '216.239.35.4' 'i.ytimg.com' '142.250.145.188'
 'fonts.gstatic.com' 'storage.googleapis.com'
 'ms-sso.copilot.microsoft.com' 'logincdn.msftauth.net' 'fpt.live.com'
 'ms-sso.copilot.com' 'browser.events.data.microsoft.com'
 'support.google.com' '108.177.127.188' 'alt1-mtalk.google.com'
 'maps.gstatic.com' 'api.login.yahoo.com' 'login.yahoo.com' 's.yimg.com'
 'udc.yahoo.com' 'region1.google-

In [9]:
import sqlite3
import pandas as pd
import tldextract
import re

GRAVITY_PATH = os.path.join(DATASET_DIR, "pihole/gravity.db")
# Step 1: Load ALL blocklisted domains directly from gravity.db
with sqlite3.connect(GRAVITY_PATH) as conn:
    gravity_df = pd.read_sql_query("SELECT domain FROM gravity", conn)

# Store as a set for O(1) ultra-fast lookup
gravity_domains = set(gravity_df['domain'].str.lower())

# Step 2: Helper function to extract root domains (ignoring IP addresses)
def get_root_domain(url):
    if pd.isna(url):
        return None
    # Skip IPv4 addresses
    if re.match(r"^\d{1,3}(\.\d{1,3}){3}$", str(url)):
        return None
    extracted = tldextract.extract(str(url))
    if extracted.domain and extracted.suffix:
        return f"{extracted.domain}.{extracted.suffix}"
    return str(url)

# Clean and prepare dynamic domain columns
df_dynamic['real_domain_clean'] = df_dynamic['real_domain'].str.lower().fillna('')
df_dynamic['root_domain'] = df_dynamic['real_domain'].apply(get_root_domain)

# Step 3: Match against Gravity using BOTH full domain and root domain
def is_blocklisted(row):
    full_domain = row['real_domain_clean']
    root_domain = row['root_domain']

    # Check exact subdomain match first, then root domain match
    if full_domain in gravity_domains:
        return True
    if root_domain and root_domain in gravity_domains:
        return True
    return False

# Apply matching logic
df_dynamic['is_tracker'] = df_dynamic.apply(is_blocklisted, axis=1)

# Step 4: Filter for caught trackers
df_dynamic_trackers = df_dynamic[df_dynamic['is_tracker']]

# Print all caught tracker domains
print("Caught Tracker Domains:")
print(df_dynamic_trackers['real_domain'].unique())

Caught Tracker Domains:
['www.googleadservices.com' 'www.googletagmanager.com'
 'ad.doubleclick.net' 'app-measurement.com'
 'pagead2.googlesyndication.com' 'browser.events.data.microsoft.com'
 'udc.yahoo.com' 'region1.google-analytics.com' 'ssl.google-analytics.com'
 'adservice.google.com']


In [10]:
import pandas as pd
import numpy as np

# 1. Standardize static dataframe column name
df_static_clean = df_static.copy()
df_dynamic_trackers = df_dynamic_trackers.copy()
df_static_clean['firmware_id'] = df_static_clean['firmware_id'].astype(str).str.strip()
if 'application.handle' in df_static_clean.columns and 'package_name' not in df_static_clean.columns:
    df_static_clean = df_static_clean.rename(columns={'application.handle': 'package_name'})


# 1. Convert to string and lowercase (just to be safe)
df_dynamic_trackers['package_name'] = df_dynamic_trackers['package_name'].astype(str).str.lower()

# 2. Split the comma-separated strings into lists
df_dynamic_trackers['package_name'] = df_dynamic_trackers['package_name'].str.split(',')

# 3. Explode the lists into separate rows
df_dynamic_trackers = df_dynamic_trackers.explode('package_name')

# 4. Strip any leading/trailing whitespace left over from the split
df_dynamic_trackers['package_name'] = df_dynamic_trackers['package_name'].str.strip()

# 5. (Optional) Filter out the "Unknown / OS" traffic
df_dynamic_trackers = df_dynamic_trackers[df_dynamic_trackers['package_name'] != 'unknown / os']

# Verify the fix
print(df_dynamic_trackers['package_name'].unique())


# 2. Aggregate network tracker activity per app (firmware_id + package_name)
dynamic_tracker_summary = df_dynamic_trackers.groupby(['firmware_id', 'package_name']).agg(
    contacted_domains=('real_domain_clean', lambda x: sorted(list(set(x)))),
    contacted_root_domains=('root_domain', lambda x: sorted(list(set(x.dropna())))),
    total_tracker_requests=('timestamp', 'count')
).reset_index()

dynamic_tracker_summary['has_network_trackers'] = True

# 3. Merge static analysis baseline with dynamic network findings
df_merged = pd.merge(
    df_static_clean,
    dynamic_tracker_summary,
    on=['firmware_id', 'package_name'],
    how='left'
)

# Fill NaNs for apps that had no tracker network requests
df_merged['has_network_trackers'] = df_merged['has_network_trackers'].fillna(False)
df_merged['total_tracker_requests'] = df_merged['total_tracker_requests'].fillna(0).astype(int)
df_merged['contacted_domains'] = df_merged['contacted_domains'].apply(lambda d: d if isinstance(d, list) else [])
df_merged['contacted_root_domains'] = df_merged['contacted_root_domains'].apply(lambda d: d if isinstance(d, list) else [])

['com.google.android.youtube' 'com.google.android.apps.youtube.music'
 'com.google.android.gms' 'com.google.android.gsf' 'com.google.android.gm'
 'com.google.android.googlequicksearchbox' 'uid 0 (os) - unknown package'
 'com.google.android.apps.recorder' 'com.google.android.apps.messaging'
 'com.google.android.apps.nbu.files'
 'com.google.android.apps.work.oobconfig' 'com.android.webview'
 'org.chromium.webview_shell' 'com.google.android.apps.maps']


In [11]:
df_merged.head()

,_id,_cls,report_date,android_app_id_reference,scanner_version,scanner_name,scan_status,error,trackers,package_name,...,os_vendor,version_detected,firmware_id,filename,partition_name,tracker_names,contacted_domains,contacted_root_domains,total_tracker_requests,has_network_trackers
0,6a4f4eb122324e493b7a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:02.498000,68b105d665e2ad36cf065848,1.3.12,Exodus,completed,None,[],com.android.bookmarkprovider,...,Google,12,68b1075d65e2ad36cf0776d5,BookmarkProvider.apk,system,[],[],[],0,False
1,6a4f4eb1fa7e8ee5127a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:01.800000,68b105d665e2ad36cf065844,1.3.12,Exodus,completed,None,[],com.android.bluetoothmidiservice,...,Google,12,68b1075d65e2ad36cf0776d5,BluetoothMidiService.apk,system,[],[],[],0,False
2,6a4f4eb256df774d197a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:03.001000,68b105d665e2ad36cf06582e,1.3.12,Exodus,completed,None,[],com.android.dreams.basic,...,Google,12,68b1075d65e2ad36cf0776d5,BasicDreams.apk,system,[],[],[],0,False
3,6a4f4eb28147f5ecd97a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:02.704000,68b105d765e2ad36cf065944,1.3.12,Exodus,completed,None,[],com.android.providers.partnerbookmarks,...,Google,12,68b1075d65e2ad36cf0776d5,PartnerBookmarksProvider.apk,system,[],[],[],0,False
4,6a4f4eb31aea578a3f7a8e3d,ApkScannerReport.ExodusReport,2026-07-09T07:33:05.799000,68b105d765e2ad36cf065892,1.3.12,Exodus,completed,None,[],com.android.htmlviewer,...,Google,12,68b1075d65e2ad36cf0776d5,HTMLViewer.apk,system,[],[],[],0,False


In [12]:
# 1. Check if df_dynamic_trackers actually has rows
print(f"Total dynamic tracker entries: {len(df_dynamic_trackers)}")

# 2. Check how many unique packages exist in both datasets
static_packages = set(df_static_clean['package_name'].dropna().astype(str).str.lower().str.strip())
dynamic_packages = set(df_dynamic_trackers['package_name'].dropna().astype(str).str.lower().str.strip())

package_overlap = static_packages.intersection(dynamic_packages)
print(f"Unique packages in static: {len(static_packages)}")
print(f"Unique packages in dynamic trackers: {len(dynamic_packages)}")
print(f"Package names that exist in BOTH datasets: {len(package_overlap)}")

# 3. Check firmware_id overlap
static_firmware = set(df_static_clean['firmware_id'].dropna().astype(str).str.strip())
dynamic_firmware = set(df_dynamic_trackers['firmware_id'].dropna().astype(str).str.strip())

firmware_overlap = static_firmware.intersection(dynamic_firmware)
print(f"\nUnique firmwares in static: {len(static_firmware)}")
print(f"Unique firmwares in dynamic trackers: {len(dynamic_firmware)}")
print(f"Firmware IDs that exist in BOTH datasets: {len(firmware_overlap)}")


Total dynamic tracker entries: 49185
Unique packages in static: 7045
Unique packages in dynamic trackers: 14
Package names that exist in BOTH datasets: 13

Unique firmwares in static: 1056
Unique firmwares in dynamic trackers: 63
Firmware IDs that exist in BOTH datasets: 63


In [13]:
# 1. Print the exact packages found in the tracker traffic
print("Unique packages in Dynamic Tracker Traffic:")
print(df_dynamic_trackers['package_name'].unique())

# 2. See how the 12,344 requests are distributed
print("\nTraffic count per package:")
print(df_dynamic_trackers['package_name'].value_counts(dropna=False))

# 3. Check if we have UIDs for traffic that is missing a package name
missing_pkg_traffic = df_dynamic_trackers[df_dynamic_trackers['package_name'].isna() | (df_dynamic_trackers['package_name'] == '') | (df_dynamic_trackers['package_name'] == 'nan')]
print(f"\nTracker requests missing a package name: {len(missing_pkg_traffic)}")
if len(missing_pkg_traffic) > 0:
    print("Top UIDs for missing packages:")
    print(missing_pkg_traffic['uid'].value_counts().head(10))

Unique packages in Dynamic Tracker Traffic:
['com.google.android.youtube' 'com.google.android.apps.youtube.music'
 'com.google.android.gms' 'com.google.android.gsf' 'com.google.android.gm'
 'com.google.android.googlequicksearchbox' 'uid 0 (os) - unknown package'
 'com.google.android.apps.recorder' 'com.google.android.apps.messaging'
 'com.google.android.apps.nbu.files'
 'com.google.android.apps.work.oobconfig' 'com.android.webview'
 'org.chromium.webview_shell' 'com.google.android.apps.maps']

Traffic count per package:
package_name
com.google.android.apps.youtube.music      13246
com.google.android.gms                      9215
com.google.android.gsf                      9215
com.google.android.youtube                  5818
com.google.android.gm                       5543
com.google.android.googlequicksearchbox     3072
com.google.android.apps.messaging           1290
com.google.android.apps.nbu.files            720
com.google.android.apps.recorder             636
org.chromium.webview

In [14]:
# Helper to check if Exodus found zero trackers
def is_static_empty(val):
    if pd.isna(val) or val is None or val == '' or val == '[]' or val == []:
        return True
    if isinstance(val, (list, np.ndarray)) and len(val) == 0:
        return True
    return False

df_merged['static_is_empty'] = df_merged['tracker_names'].apply(is_static_empty)

# Identify Complete False Negatives
complete_false_negatives = df_merged[
    (df_merged['has_network_trackers'] == True) &
    (df_merged['static_is_empty'] == True)
].copy()

print(f"Total Complete False Negatives found: {len(complete_false_negatives)}")

# Display the top stealth apps generating tracker traffic missed by Exodus
cols_to_show = [
    'firmware_id',
    'package_name',
    'application.name',
    'contacted_root_domains',
    'total_tracker_requests'
]

display(complete_false_negatives[cols_to_show].sort_values(by='total_tracker_requests', ascending=False).head(100))

Total Complete False Negatives found: 57


,firmware_id,package_name,application.name,contacted_root_domains,total_tracker_requests
16285,68b24edf65e2ad36cf088805,com.google.android.gsf,Google Services Framework,"[app-measurement.com, google-analytics.com, go...",448
29882,6986923fab2ebd4d067483a5,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googletagmanager.com]",412
117245,69869a4fab2ebd4d067b71f9,com.google.android.gsf,Google Services Framework,"[app-measurement.com, google-analytics.com, go...",303
34203,69895f69ab2ebd4d0620d788,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googleadservices.com, go...",294
108638,69832d4770168e916d8d54d4,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googleadservices.com, go...",285
22443,68b2ebc265e2ad36cf7e81e2,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googletagmanager.com]",268
141115,698f110ee0e6709b7bc0f8fd,com.google.android.gsf,Google Services Framework,"[app-measurement.com, google-analytics.com, go...",268
26384,698322b970168e916d8324f8,com.google.android.gsf,Google Services Framework,"[app-measurement.com, google-analytics.com, go...",267
32403,698c1bb3ab2ebd4d06d3481f,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googleadservices.com, go...",257
18993,68b1e6f865e2ad36cfb85aaa,com.google.android.gsf,Google Services Framework,"[app-measurement.com, googleadservices.com, go...",256


In [15]:
# Verification Step: Check the raw Exodus output for these false negatives
# 1. Isolate the keys (firmware and package) of the false negatives we found
keys_to_verify = complete_false_negatives[['firmware_id', 'package_name']].drop_duplicates()

# 2. Do an exact lookup (inner join) against your cleaned static dataset
# (Make sure to use df_static_clean so the keys match perfectly)
verification_df = pd.merge(
    df_static_clean,
    keys_to_verify,
    on=['firmware_id', 'package_name'],
    how='inner'
)

# 3. Print the raw 'tracker_names' directly from the static dataset
print("=== VERIFYING RAW EXODUS OUTPUT FOR FALSE NEGATIVES ===")
display_cols = ['firmware_id', 'package_name', 'tracker_names']

# We use to_markdown() or display() to ensure we see the raw string/list representations clearly
display(verification_df)

=== VERIFYING RAW EXODUS OUTPUT FOR FALSE NEGATIVES ===


,_id,_cls,report_date,android_app_id_reference,scanner_version,scanner_name,scan_status,error,trackers,package_name,...,application.permissions,application.libraries,apk.path,apk.checksum,os_vendor,version_detected,firmware_id,filename,partition_name,tracker_names
0,6a4f4f9912aba6b9f87a8e41,ApkScannerReport.ExodusReport,2026-07-09T07:36:57.406000,68b1063265e2ad36cf06a68a,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""android.permission.MANAGE_USERS"",""com.google...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b1075d65e2ad36cf0776d5,GoogleServicesFramework.apk,system_ext,[]
1,6a4f6be45a65a0466e8b942a,ApkScannerReport.ExodusReport,2026-07-09T09:37:40.594000,68b086e565e2ad36cfaa4c96,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""android.permission.REBOOT"",""android.permissi...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,0c80d0321eadf7a1d5132ecf1087f4ec5478e8db5a4409...,Google,12,68b0892a65e2ad36cfab2516,GoogleServicesFramework.apk,system_ext,[]
2,6a4f6f198e6db313c128ffc4,ApkScannerReport.ExodusReport,2026-07-09T09:51:21.016000,68b0d90165e2ad36cfe0af42,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""android.permission.INVOKE_CARRIER_SETUP"",""an...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b0da2165e2ad36cfe19b3d,GoogleServicesFramework.apk,system_ext,[]
3,6a4f749aed5065c9c9fdba6a,ApkScannerReport.ExodusReport,2026-07-09T10:14:49.997000,68b17b1465e2ad36cf6272f4,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""android.permission.RECEIVE_DATA_ACTIVITY_CHA...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b17c4d65e2ad36cf638567,GoogleServicesFramework.apk,system_ext,[]
4,6a4f753f90afd68819f0f4b8,ApkScannerReport.ExodusReport,2026-07-09T10:17:34.700000,68b1250365e2ad36cf1f37a9,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""com.android.vending.TOS_ACKED"",""android.perm...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b1266865e2ad36cf200744,GoogleServicesFramework.apk,system_ext,[]
5,6a4f7a0d85cb8e94cbf0f4e0,ApkScannerReport.ExodusReport,2026-07-09T10:38:03.797000,68b1332465e2ad36cf2a3afb,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""com.android.vending.TOS_ACKED"",""android.perm...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b134be65e2ad36cf2b556f,GoogleServicesFramework.apk,system_ext,[]
6,6a4f7fb13bf6da1fd3dc7aeb,ApkScannerReport.ExodusReport,2026-07-09T11:02:06.695000,68b1f71065e2ad36cfc42ed3,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""android.permission.RECOVERY"",""android.permis...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b1f86e65e2ad36cfc530f4,GoogleServicesFramework.apk,system_ext,[]
7,6a4f84a4b2bddb462211c8d6,ApkScannerReport.ExodusReport,2026-07-09T11:23:16.098000,68b24d8965e2ad36cf07920b,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""com.android.vending.TOS_ACKED"",""android.perm...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage/dc63ac8b-3...,f013a38cf9e44112508484a21fda6531b076d8b15f1cd1...,Google,12,68b24edf65e2ad36cf088805,GoogleServicesFramework.apk,system_ext,[]
8,6a4f85990d7d31401011c8de,ApkScannerReport.ExodusReport,2026-07-09T11:27:21.103000,68b24e0865e2ad36cf07f919,1.3.12,Exodus,completed,None,[],com.google.android.gsf,...,"[""com.android.vending.TOS_ACKED"",""android.perm...","[""org.apache.http.legacy""]",/var/www/file_store/01_file_storage

In [16]:
# Helper to check if contacted root domains are reflected in Exodus tracker names
def find_unlisted_domains(row):
    static_trackers = str(row['tracker_names']).lower() if not pd.isna(row['tracker_names']) else ""
    contacted_roots = row['contacted_root_domains']

    unlisted = []
    for domain in contacted_roots:
        # Extract main brand (e.g., "doubleclick" from "doubleclick.net", "facebook" from "facebook.com")
        brand = domain.split('.')[0]
        if brand not in static_trackers:
            unlisted.append(domain)

    return unlisted

# Apply only to apps where Exodus found SOME trackers and network traffic occurred
df_merged['unlisted_contacted_domains'] = df_merged.apply(find_unlisted_domains, axis=1)

partial_false_negatives = df_merged[
    (df_merged['has_network_trackers'] == True) &
    (df_merged['static_is_empty'] == False) &
    (df_merged['unlisted_contacted_domains'].apply(len) > 0)
].copy()

print(f"Total Partial False Negatives found: {len(partial_false_negatives)}")

display(partial_false_negatives[[
    'firmware_id', 'package_name', 'tracker_names', 'unlisted_contacted_domains', 'total_tracker_requests'
]].head(15))

Total Partial False Negatives found: 236


,firmware_id,package_name,tracker_names,unlisted_contacted_domains,total_tracker_requests
1198,68b1075d65e2ad36cf0776d5,com.google.android.youtube,"[""Google Firebase Analytics"",""Google AdMob""]","[googleadservices.com, googlesyndication.com]",39
1283,68b1075d65e2ad36cf0776d5,com.google.android.gms,"[""Google CrashLytics"",""Google Analytics"",""Goog...","[app-measurement.com, google-analytics.com, go...",150
1349,68b1075d65e2ad36cf0776d5,com.google.android.googlequicksearchbox,"[""Google Analytics"",""Google Firebase Analytics...","[googleadservices.com, googletagmanager.com]",169
9943,68b0892a65e2ad36cfab2516,com.google.android.apps.youtube.music,"[""Google Firebase Analytics"",""Google AdMob""]","[doubleclick.net, googleadservices.com, google...",79
11258,68b0892a65e2ad36cfab2516,com.google.android.youtube,"[""Google Firebase Analytics"",""Google AdMob""]",[googleadservices.com],70
12487,68b0da2165e2ad36cfe19b3d,com.google.android.youtube,"[""Google Firebase Analytics"",""Google AdMob""]",[googlesyndication.com],38
13265,68b134be65e2ad36cf2b556f,com.google.android.gms,"[""Google CrashLytics"",""Google Analytics"",""Goog...","[app-measurement.com, googletagmanager.com]",193
13428,68b0892a65e2ad36cfab2516,com.google.android.gms,"[""Google CrashLytics"",""Google Analytics"",""Goog...","[app-measurement.com, googlesyndication.com]",61
13438,68b17c4d65e2ad36cf638567,com.google.android.gm,"[""Google Analytics""]","[google-analytics.com, googletagmanager.com, m...",157
13920,68b1266865e2ad36cf200744,com.google.android.youtube,"[""Google Firebase Analytics"",""Google AdMob""]","[googleadservices.com, googlesyndication.com]",68


In [18]:
# ==========================================
# 1. CATEGORIZE THE APPS
# ==========================================
def categorize_app(row):
    if row['has_network_trackers'] and not row['static_is_empty']:
        return 'True Positive (Active Detected)'
    elif row['has_network_trackers'] and row['static_is_empty']:
        return 'False Negative (Complete Miss)'
    elif not row['has_network_trackers'] and row['static_is_empty']:
        return 'True Negative (Clean)'
    else:
        return 'Static Only (Dormant/Unused Tracker)'

# Apply the categorization to our merged dataset
df_merged['analysis_category'] = df_merged.apply(categorize_app, axis=1)

# ==========================================
# 2. FILTER FOR EXECUTED APPS ONLY
# ==========================================
# Clean the dynamic columns for the baseline extraction
df_dynamic['firmware_id_clean'] = df_dynamic['firmware_id'].astype(str).str.split('_').str[0].str.strip()
df_dynamic['package_name_clean'] = df_dynamic['package_name'].astype(str).str.lower().str.split(',')
df_dynamic_exploded = df_dynamic.explode('package_name_clean')
df_dynamic_exploded['package_name_clean'] = df_dynamic_exploded['package_name_clean'].str.strip()

# Create a deduplicated list of actually executed apps
executed_apps = df_dynamic_exploded[['firmware_id_clean', 'package_name_clean']].drop_duplicates()
executed_apps = executed_apps.rename(columns={
    'firmware_id_clean': 'firmware_id',
    'package_name_clean': 'package_name'
})

# Filter df_merged to ONLY include these executed apps
df_evaluated = pd.merge(
    executed_apps,
    df_merged,
    on=['firmware_id', 'package_name'],
    how='inner'
)

In [19]:
# ==========================================
# 3. RECALCULATE STATS & EXPORT
# ==========================================
summary_stats_executed = df_evaluated['analysis_category'].value_counts().reset_index()
summary_stats_executed.columns = ['Category', 'App Count']
summary_stats_executed['Percentage'] = (summary_stats_executed['App Count'] / len(df_evaluated) * 100).round(2).astype(str) + '%'

# --- NEW PRINTS FOR TRACE TOTALS ---
total_trace_instances = len(executed_apps)
total_unique_trace_packages = executed_apps['package_name'].nunique()

print(f"Total app instances with network traces (firmware + package): {total_trace_instances}")
print(f"Total unique app packages with network traces: {total_unique_trace_packages}")
# -----------------------------------

print(f"Total apps actually executed and evaluated (in both datasets): {len(df_evaluated)}")
print("=== CORRECTED METRICS (EXECUTED APPS ONLY) ===")
display(summary_stats_executed)

# Generate the corrected LaTeX table with booktabs
latex_metrics_corrected = (
    summary_stats_executed.style
    .hide(axis="index")
    .to_latex(
        caption="Classification Performance of Static Tracker Detection vs Dynamic Traffic (Executed Apps Only)",
        label="tab:fn_analysis_results",
        column_format="lrr",
        hrules=True
    )
)
print("\nCorrected LaTeX Table:")
print(latex_metrics_corrected)

Total app instances with network traces (firmware + package): 2749
Total unique app packages with network traces: 54
Total apps actually executed and evaluated (in both datasets): 2070
=== CORRECTED METRICS (EXECUTED APPS ONLY) ===


,Category,App Count,Percentage
0,True Negative (Clean),1220,58.94%
1,Static Only (Dormant/Unused Tracker),553,26.71%
2,True Positive (Active Detected),240,11.59%
3,False Negative (Complete Miss),57,2.75%



Corrected LaTeX Table:
\begin{table}
\caption{Classification Performance of Static Tracker Detection vs Dynamic Traffic (Executed Apps Only)}
\label{tab:fn_analysis_results}
\begin{tabular}{lrr}
\toprule
Category & App Count & Percentage \\
\midrule
True Negative (Clean) & 1220 & 58.94% \\
Static Only (Dormant/Unused Tracker) & 553 & 26.71% \\
True Positive (Active Detected) & 240 & 11.59% \\
False Negative (Complete Miss) & 57 & 2.75% \\
\bottomrule
\end{tabular}
\end{table}



In [20]:
import pandas as pd

# Temporarily force pandas to print EVERYTHING without truncating columns or rows
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# =========================================================
# 1. COMPLETE FALSE NEGATIVES: All Contacted Domains
# =========================================================
print("=== COMPLETE FALSE NEGATIVES: All Contacted Domains ===")
print("(Exodus found NO trackers, but these domains were contacted)")

# Select the relevant columns
cfn_domains = complete_false_negatives[['package_name', 'application.name', 'contacted_root_domains']].copy()

# Explode the list so each contacted domain gets its own row alongside the package name
cfn_exploded = cfn_domains.explode('contacted_root_domains').rename(
    columns={'contacted_root_domains': 'contacted_domain'}
)

# Sort alphabetically by package name, then by domain
cfn_exploded = cfn_exploded.sort_values(by=['package_name', 'contacted_domain']).dropna(subset=['contacted_domain'])
display(cfn_exploded)


# =========================================================
# 2. COMPLETE FALSE NEGATIVES: Static Analysis Verification
# =========================================================
print("\n=== COMPLETE FALSE NEGATIVES: Static Analysis Verification ===")
print("(Verifying that the static analysis output for these apps was indeed empty)")

cfn_static = complete_false_negatives[['package_name', 'tracker_names']]
display(cfn_static)


# =========================================================
# 3. PARTIAL FALSE NEGATIVES: Static Trackers vs Network Traffic
# =========================================================
print("\n=== PARTIAL FALSE NEGATIVES: Static Trackers vs Contacted Domains ===")
print("(Exodus found SOME trackers, but missed the ones causing these specific domains)")

# Select the fields to compare what Exodus knew vs. what actually happened
pfn_subset = partial_false_negatives[[
    'package_name',
    'application.name',
    'tracker_names',
    'unlisted_contacted_domains'
]].copy()

pfn_subset = pfn_subset.sort_values(by='package_name')
display(pfn_subset)

# Reset pandas display options back to their defaults so it doesn't affect the rest of your notebook
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')

=== COMPLETE FALSE NEGATIVES: All Contacted Domains ===
(Exodus found NO trackers, but these domains were contacted)


,package_name,application.name,contacted_domain
209,com.google.android.gsf,Google Services Framework,app-measurement.com
8935,com.google.android.gsf,Google Services Framework,app-measurement.com
10040,com.google.android.gsf,Google Services Framework,app-measurement.com
11842,com.google.android.gsf,Google Services Framework,app-measurement.com
12066,com.google.android.gsf,Google Services Framework,app-measurement.com
13490,com.google.android.gsf,Google Services Framework,app-measurement.com
15314,com.google.android.gsf,Google Services Framework,app-measurement.com
16285,com.google.android.gsf,Google Services Framework,app-measurement.com
16609,com.google.android.gsf,Google Services Framework,app-measurement.com
18993,com.google.android.gsf,Google Services Framework,app-measurement.com



=== COMPLETE FALSE NEGATIVES: Static Analysis Verification ===
(Verifying that the static analysis output for these apps was indeed empty)


,package_name,tracker_names
209,com.google.android.gsf,[]
8935,com.google.android.gsf,[]
10040,com.google.android.gsf,[]
11842,com.google.android.gsf,[]
12066,com.google.android.gsf,[]
13490,com.google.android.gsf,[]
15314,com.google.android.gsf,[]
16285,com.google.android.gsf,[]
16609,com.google.android.gsf,[]
18993,com.google.android.gsf,[]



=== PARTIAL FALSE NEGATIVES: Static Trackers vs Contacted Domains ===
(Exodus found SOME trackers, but missed the ones causing these specific domains)


,package_name,application.name,tracker_names,unlisted_contacted_domains
82555,com.google.android.apps.maps,Maps,"[""Google Firebase Analytics""]",[app-measurement.com]
39285,com.google.android.apps.maps,Maps,"[""Google Firebase Analytics""]",[app-measurement.com]
121818,com.google.android.apps.messaging,Messages,"[""Google Firebase Analytics""]",[app-measurement.com]
144475,com.google.android.apps.messaging,Messages,"[""Google Firebase Analytics""]",[app-measurement.com]
144506,com.google.android.apps.messaging,Messages,"[""Google Firebase Analytics""]",[app-measurement.com]
141920,com.google.android.apps.messaging,Messages,"[""Google Firebase Analytics""]",[app-measurement.com]
144367,com.google.android.apps.messaging,Messages,"[""Google Firebase Analytics""]",[app-measurement.com]
143159,com.google.android.apps.nbu.files,Files by Google,"[""Google Firebase Analytics""]",[app-measurement.com]
119397,com.google.android.apps.nbu.files,Files by Google,"[""Google Firebase Analytics""]",[app-measurement.com]
141711,com.google.android.apps.nbu.files,Files by Google,"[""Google Firebase Analytics""]",[app-measurement.com]


In [21]:
import pandas as pd

# Define the package names to look for (including the likely correct spelling)
target_packages = ["com.goog.android.gsf", "com.google.android.gsf"]

# Filter the cleaned static dataframe
gsf_static_check = df_static_clean[
    df_static_clean['package_name'].isin(target_packages)
].copy()

# Select the most relevant columns to display
cols_to_view = ['firmware_id', 'package_name', 'tracker_names']

# Ensure the columns actually exist before selecting them
cols_to_view = [c for c in cols_to_view if c in gsf_static_check.columns]

print(f"Found {len(gsf_static_check)} instances of the GSF package in the static analysis data.\n")

# Use a context manager to temporarily show the full row/column contents without truncation
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(gsf_static_check[cols_to_view])

Found 924 instances of the GSF package in the static analysis data.



,firmware_id,package_name,tracker_names
index,,,
1169,68b1075d65e2ad36cf0776d5,com.google.android.gsf,[]
2393,68af849165e2ad36cfd4a6d0,com.google.android.gsf,[]
2646,68af110965e2ad36cf7475cd,com.google.android.gsf,[]
3007,68af61ae65e2ad36cfb8b700,com.google.android.gsf,[]
3087,68afbec865e2ad36cf05f3b3,com.google.android.gsf,[]
3406,68af527e65e2ad36cfab0359,com.google.android.gsf,[]
3872,68afb0cd65e2ad36cffa1f06,com.google.android.gsf,[]
4009,68af219965e2ad36cf8260b2,com.google.android.gsf,[]
4215,68af388565e2ad36cf94dea9,com.google.android.gsf,[]
